# 20 (DE) — UDF Authoring & Function Registry

**Data Engineer perspective.** IrisPark executes functions at three tiers: native SQL pushdown, ObjectScript UDF packs, and Embedded Python (EPython). This notebook maps the tiers, installs the packs, registers a custom Python function, and reads the registry that powers `explain(extended=True)`.

**UDC note**: EPython functions live in `irispark_udc.py` on the IRIS server filesystem. Its absolute path is configured via the `IRISPARK_UDC_PATH` environment variable (the Docker setup here sets `/external/durable/mgr/python/irispark_udc.py`). Without it, sessions warn and skip only that tier.

In [ ]:
import os
from dotenv import load_dotenv
from irispark import IrisParkSession

load_dotenv()

# Connection via environment variables (matches examples/basic_usage.py).
# Set IRIS_HOST / IRIS_PORT / IRIS_NAMESPACE / IRIS_USERNAME / IRIS_PASSWORD.
try:
    session = IrisParkSession.builder() \
        .host(os.environ.get("IRIS_HOST", "localhost")) \
        .port(int(os.environ.get("IRIS_PORT", 1972))) \
        .namespace(os.environ.get("IRIS_NAMESPACE", "USER")) \
        .username(os.environ.get("IRIS_USERNAME", "_SYSTEM")) \
        .password(os.environ.get("IRIS_PASSWORD", "SYS")) \
        .getOrCreate()
    print("Connected to IRIS:", session)
except Exception as e:
    print("SKIP: IRIS not reachable -", e)
    session = None

In [ ]:
if session is None:
    raise SystemExit("IRIS not reachable; skipping this notebook.")

## 1. Tier 1 — native SQL pushdown

Plain SQL built-ins compile straight into the statement; no registration exists at all.

In [ ]:
import pandas as pd
from irispark.functions import upper, sqrt, abs as fabs

df = session.createDataFrame(pd.DataFrame({"nome": ["ana"], "x": [9.0]}))
df.select(upper("nome").alias("up"), sqrt("x").alias("raiz"), fabs("x").alias("abs")).show()
print(df.select(sqrt("x")).to_sql())

## 2. Tier 2 — ObjectScript UDF packs

Installed automatically on session start (idempotent DDL). Example: `levenshtein` distance and trigonometry run as ObjectScript routines.

In [ ]:
from irispark.functions import levenshtein, cos, lit

df2 = session.createDataFrame(pd.DataFrame({"a": ["kitten"], "b": ["sitting"]}))
df2.select(levenshtein("a", "b").alias("dist"), cos(lit(1.0)).alias("cos1")).show()

## 3. Tier 3 — Embedded Python pack

When `IRISPARK_UDC_PATH` resolves on the server, format-style EPython UDFs install too. The session log tells you which tier loaded.

In [ ]:
import os
print("IRISPARK_UDC_PATH =", os.environ.get("IRISPARK_UDC_PATH", "(not set)"))
from irispark.functions import sha2

df3 = session.createDataFrame(pd.DataFrame({"s": ["iris"]}))
df3.select("s", sha2("s", 256).alias("sha256")).show()

## 4. Custom Python function — `session.udf.register`

Registers into the function registry used by plan resolution.

In [ ]:
session.udf.register("tamanho", lambda s_: len(s_))
print("registered:", session.udf._get("tamanho")("irispark"))

## 5. Registry inventory

`list_functions` is what `explain(extended=True)` prints — grouped by execution engine.

In [ ]:
from irispark.registry import list_functions

funcs = list_functions()
print("total registered:", len(funcs))
by_exec = {}
for f in funcs:
    by_exec[f.execution] = by_exec.get(f.execution, 0) + 1
for k, v in sorted(by_exec.items()):
    print(f"  {k}: {v}")

## 6. Engine mapping for a mixed query

One select touching all three tiers — `explain` shows who executes what.

In [ ]:
from irispark.functions import median

df4 = session.createDataFrame(pd.DataFrame({"x": [1.0, 2.0, 3.0, 4.0]}))
mixed = df4.select(upper("x").alias("u"), median("x").alias("m"), levenshtein("x", "x").alias("l"))
mixed.explain(extended=True)

In [ ]:
if session is not None:
    session.close()
    print("Session closed.")